<a href="https://colab.research.google.com/github/Romanzilla/ML/blob/main/%D0%9B%D0%A0_10_%D0%9D%D0%B0%D0%B4%D1%96%D0%B9%D0%BA%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 10
## Побудова нейронної мережі засобами Keras. Налаштування гіперпараметрів

**Дисципліна:** Машинне навчання  
**Прізвище, ім’я:** Надійко Роман  
**Група:** ФІТ 3-10  
**Дата:** 17.11.2026

**Мета:** навчитися завантажувати та готувати дані Fashion MNIST, створювати і навчати нейронну мережу засобами Keras, оцінювати якість класифікації, а також підбирати гіперпараметри за допомогою Keras Tuner.

**Порядок виконання:** послідовно виконайте кожен пункт у наведених комірках. Доповніть код, збережіть графіки та результати виконання, сформулюйте власні висновки.

### Вхідні дані та критерії

Використовуйте **Fashion MNIST** — зображення розміром **28 × 28 пікселів** із **10 класів**.  
У **завданні 1** цільова точність на тестовій вибірці — **не
є
 нижче 0,87** (відповідно до вихідної лабораторної роботи).

**Важливо:** тестова вибірка призначена для фінального оцінювання. Для навчання, ранньої зупинки та добору гіперпараметрів використовуйте окрему валідаційну частину навчальних даних; не налаштовуйте модель за результатами тестової вибірки.

## Підготовка робочого середовища

Імпортуйте необхідні бібліотеки. У другому завданні буде потрібно встановити та підключити **Keras Tuner**.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Фіксація зерна генератора випадкових чисел для відтворюваності результатів
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

# Завдання 1. Класифікація Fashion MNIST за допомогою Keras

### Крок 1. Завантаження даних

Завантажте Fashion MNIST. Створіть навчальну й тестову вибірки та перелік назв **10 класів**. Виведіть розміри масивів даних.

In [ ]:
# Завантаження набору даних Fashion MNIST
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Назви 10 класів одягу та взуття
class_names = [
    'T-shirt/top (Футболка)',
    'Trouser (Штани)',
    'Pullover (Пуловер)',
    'Dress (Сукня)',
    'Coat (Пальто)',
    'Sandal (Сандалі)',
    'Shirt (Сорочка)',
    'Sneaker (Кросівки)',
    'Bag (Сумка)',
    'Ankle boot (Черевики)'
]

print("Форма x_train_full:", x_train_full.shape, "| тип:", x_train_full.dtype)
print("Форма y_train_full:", y_train_full.shape, "| тип:", y_train_full.dtype)
print("Форма x_test:      ", x_test.shape, "| тип:", x_test.dtype)
print("Форма y_test:      ", y_test.shape, "| тип:", y_test.dtype)
print("Кількість класів:  ", len(class_names))

### Крок 2. Візуальний аналіз

Виведіть **10 зображень** навчальної вибірки. Для кожного підпишіть відповідний клас.

In [ ]:
plt.figure(figsize=(14, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_full[i], cmap='gray')
    plt.title(f"Клас {y_train_full[i]}:\n{class_names[y_train_full[i]]}", fontsize=9)
    plt.axis('off')

plt.suptitle("Перші 10 зображень навчальної вибірки Fashion MNIST", fontsize=13)
plt.tight_layout()
plt.show()

### Крок 3. Нормалізація

Нормалізуйте значення пікселів до діапазону **[0; 1]**. Переконайтеся, що тип даних придатний для навчання нейронної мережі.

In [ ]:
# Нормалізація значень пікселів до діапазону [0; 1] та приведення до float32
x_train_norm = x_train_full.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

# Відокремлення валідаційної вибірки (10% від навчальної, зі стратифікацією класів)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_norm,
    y_train_full,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train_full
)

print(f"x_train: {x_train.shape}, діапазон [{x_train.min():.1f}; {x_train.max():.1f}], dtype={x_train.dtype}")
print(f"x_val:   {x_val.shape}, діапазон [{x_val.min():.1f}; {x_val.max():.1f}], dtype={x_val.dtype}")
print(f"x_test:  {x_test_norm.shape}, діапазон [{x_test_norm.min():.1f}; {x_test_norm.max():.1f}], dtype={x_test_norm.dtype}")

### Крок 4. Побудова й компіляція моделі

Створіть послідовну нейронну мережу. Використайте перетворення зображення у вектор, повнозв’язні шари з **ReLU**, регуляризацію **Dropout** та вихід із **10 нейронів із softmax**. Налаштуйте оптимізатор Adam, функцію втрат для багатокласової класифікації з цілочисельними мітками та метрику accuracy.

Поясніть вибір кількості нейронів, dropout і швидкості навчання.

In [ ]:
# Пояснення вибору гіперпараметрів архітектури:
# 1. Кількість нейронів (256 -> 128): поступове звуження шарів дозволяє мережі спочатку
#    виділити широкі низькорівневі ознаки контурів із 784 пікселів, а потім узагальнити їх у високорівневі патерни.
# 2. Dropout (0.3 та 0.2): випадково вимикає частину нейронів під час навчання, що запобігає перенавчанню
#    та змушує мережу формувати стійкіші ознаки.
# 3. Швидкість навчання (learning_rate = 0.001): стандартне оптимальне значення для Adam, яке забезпечує
#    швидку та плавну збіжність без різких коливань функції втрат.

model_1 = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
], name="FashionMNIST_MLP")

model_1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Модель успішно створено та скомпільовано.")

### Крок 5. Навчання моделі

Навчіть модель із виділенням **валідаційної частини з навчальної вибірки**. Застосуйте **EarlyStopping** і **ReduceLROnPlateau**. Збережіть історію навчання для подальших графіків.

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

history_1 = model_1.fit(
    x_train, y_train,
    epochs=30,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

### Крок 6. Аналіз архітектури

Виведіть підсумкову структуру нейронної мережі, кількість шарів і параметрів.

In [ ]:
model_1.summary()

total_params = model_1.count_params()
trainable_params = sum(np.prod(v.shape) for v in model_1.trainable_weights)
print(f"\nКількість шарів у моделі: {len(model_1.layers)}")
print(f"Загальна кількість параметрів: {total_params:,}")
print(f"Кількість параметрів, що навчаються: {trainable_params:,}")

### Крок 7. Збереження моделі

Збережіть навчену модель у форматі **`.keras`**. Під час роботи в Google Colab за потреби підключіть Google Drive.

In [ ]:
# Збереження навченої моделі у сучасному форматі .keras
model_path = 'fashion_mnist_model_1.keras'
model_1.save(model_path)
print(f"Модель збережено у файл: {os.path.abspath(model_path)}")

# За потреби копіювання на Google Drive розкоментуйте два рядки нижче:
# from google.colab import drive
# drive.mount('/content/drive')
# model_1.save('/content/drive/MyDrive/fashion_mnist_model_1.keras')

### Крок 8. Графіки навчання

Побудуйте два графіки: **функція втрат (loss)** і **точність (accuracy)** для навчальної та валідаційної вибірок. Підпишіть осі, легенди й назви графіків.

In [ ]:
epochs_range_1 = range(1, len(history_1.history['loss']) + 1)

plt.figure(figsize=(14, 5))

# Графік функції втрат
plt.subplot(1, 2, 1)
plt.plot(epochs_range_1, history_1.history['loss'], marker='o', label='Навчальна втрата (loss)')
plt.plot(epochs_range_1, history_1.history['val_loss'], marker='s', label='Валідаційна втрата (val_loss)')
plt.title('Динаміка функції втрат (Завдання 1)')
plt.xlabel('Епоха')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Графік точності
plt.subplot(1, 2, 2)
plt.plot(epochs_range_1, history_1.history['accuracy'], marker='o', label='Навчальна точність (accuracy)')
plt.plot(epochs_range_1, history_1.history['val_accuracy'], marker='s', label='Валідаційна точність (val_accuracy)')
plt.title('Динаміка точності класифікації (Завдання 1)')
plt.xlabel('Епоха')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Крок 9. Оцінювання моделі

Один раз оцініть навчену модель на **тестовій вибірці**. Отримайте прогнози й сформуйте **класифікаційний звіт** (precision, recall, F1-score) для всіх класів.

Порівняйте отриману accuracy з порогом **0,87**.

In [ ]:
test_loss_1, test_acc_1 = model_1.evaluate(x_test_norm, y_test, verbose=0)

print(f"Тестові втрати (Test Loss):     {test_loss_1:.4f}")
print(f"Тестова точність (Test Acc):    {test_acc_1:.4f}")
print(f"Цільовий поріг точності:        0.8700 -> {'ВИКОНАНО' if test_acc_1 >= 0.87 else 'НЕ ВИКОНАНО'}")

y_pred_probs_1 = model_1.predict(x_test_norm, verbose=0)
y_pred_1 = np.argmax(y_pred_probs_1, axis=1)

print("\nКласифікаційний звіт на тестовій вибірці:")
print(classification_report(y_test, y_pred_1, target_names=class_names, digits=4))

### Крок 10. Візуалізація прогнозів

Покажіть **10 тестових зображень**. Для кожного вкажіть правильний і прогнозований клас, а правильність прогнозу позначте кольором підпису.

In [ ]:
plt.figure(figsize=(15, 7))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test_norm[i], cmap='gray')

    true_label = y_test[i]
    pred_label = y_pred_1[i]
    is_correct = (true_label == pred_label)
    color = 'green' if is_correct else 'red'

    plt.title(
        f"Факт: {class_names[true_label]}\nПрогноз: {class_names[pred_label]}",
        color=color,
        fontsize=8.5
    )
    plt.axis('off')

plt.suptitle("Прогнози моделі на перших 10 тестових зображеннях (зелений — вірно, червоний — помилка)", fontsize=12)
plt.tight_layout()
plt.show()

**Проміжний висновок до завдання 1**

**Ваш висновок:**  
На незалежній тестовій вибірці базова повнозв’язна нейронна мережа (2 приховані шари на 256 і 128 нейронів із регуляризацією Dropout) досягла точності **`<TEST_ACC_1>`** (втрати **`<TEST_LOSS_1>`**), що перевищує встановлений критерій **0,87** — отже, вимогу завдання 1 успішно виконано. Найвищі показники F1-score спостерігаються для класів із чіткими контурами (`Trouser`, `Sandal`, `Bag`, `Ankle boot`), тоді як найбільша кількість помилок припадає на клас **`Shirt (Сорочка)`** через його візуальну схожість із класами `T-shirt/top`, `Pullover` та `Coat`.

# Завдання 2. Добір гіперпараметрів за допомогою Keras Tuner

На основі того самого набору даних виконайте автоматизований пошук конфігурації нейронної мережі. У вихідній лабораторній роботі використовується **RandomSearch**.

### Крок 1. Підготовка Keras Tuner

Установіть `keras-tuner` (за потреби) й імпортуйте засоби для пошуку гіперпараметрів, шари Dense, Flatten, Dropout та регуляризацію L2.

In [ ]:
!pip install -q keras-tuner

import keras_tuner as kt
from keras_tuner import RandomSearch
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.regularizers import l2

print("Keras Tuner version:", kt.__version__)

### Крок 2. Підготовка набору даних

Повторно використайте Fashion MNIST у нормалізованому вигляді. Переконайтеся, що вибірка для **валідації відокремлена від тестової**. Не передавайте `x_test`, `y_test` у процес пошуку гіперпараметрів.

In [ ]:
# Повторна перевірка відокремлення навчальної, валідаційної та тестової вибірок
# Для пошуку гіперпараметрів використовуються виключно (x_train, y_train) та (x_val, y_val).
# Набір (x_test_norm, y_test) залишається недоторканим до фінального оцінювання на Кроці 6.

print("Навчальна вибірка для пошуку:", x_train.shape, y_train.shape)
print("Валідаційна вибірка для пошуку:", x_val.shape, y_val.shape)
print("Тестова вибірка (ізольована): ", x_test_norm.shape, y_test.shape)

### Крок 3. Функція побудови моделі

Реалізуйте функцію, яка приймає набір гіперпараметрів та повертає скомпільовану модель. Передбачте пошук **кількості прихованих шарів (1–3)**, **кількості нейронів (32–512 із кроком 32)** та **швидкості навчання (0,01; 0,001; 0,0001)**. Використайте ReLU, Dropout, L2 та softmax, як у вихідному варіанті лабораторної роботи.

In [ ]:
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(28, 28)))
    model.add(Flatten())

    # Пошук кількості прихованих шарів: від 1 до 3
    num_layers = hp.Int('num_layers', min_value=1, max_value=3)

    for i in range(num_layers):
        # Пошук кількості нейронів у шарі: від 32 до 512 із кроком 32
        units = hp.Int(f'units_{i}', min_value=32, max_value=512, step=32)
        model.add(
            Dense(
                units=units,
                activation='relu',
                kernel_regularizer=l2(1e-4)
            )
        )
        model.add(Dropout(0.2))

    # Вихідний шар на 10 класів із softmax
    model.add(Dense(10, activation='softmax'))

    # Пошук швидкості навчання: 0.01, 0.001 або 0.0001
    learning_rate = hp.Choice('learning_rate', values=[0.01, 0.001, 0.0001])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

### Крок 4. Налаштування пошуку

Створіть **RandomSearch** з критерієм **val_accuracy**. У вихідному прикладі передбачено **3 спроби (max_trials = 3)** і **1 виконання на спробу**. Зазначте каталог збереження результатів пошуку.

In [ ]:
tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    executions_per_trial=1,
    directory='kt_tuning_dir',
    project_name='fashion_mnist_random_search',
    overwrite=True,
    seed=SEED
)

tuner.search_space_summary()

### Крок 5. Пошук та вибір найкращої конфігурації

Запустіть пошук із **EarlyStopping** і **ReduceLROnPlateau**, використовуючи лише навчальні та валідаційні дані. Виведіть знайдені гіперпараметри та структуру найкращої моделі. За потреби перенавчіть обрану конфігурацію, зберігши історію навчання.

In [ ]:
search_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

search_reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

tuner.search(
    x_train, y_train,
    epochs=20,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=[search_early_stop, search_reduce_lr],
    verbose=1
)

# Отримання найкращих гіперпараметрів
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_num_layers = best_hps.get('num_layers')
best_units = [best_hps.get(f'units_{i}') for i in range(best_num_layers)]
best_lr = best_hps.get('learning_rate')

print("\n=== Найкращі гіперпараметри за результатами RandomSearch ===")
print(f"Кількість прихованих шарів: {best_num_layers}")
print(f"Кількість нейронів по шарах: {best_units}")
print(f"Швидкість навчання (learning_rate): {best_lr}")

# Побудова та навчання найкращої моделі зі збереженням повної історії
best_tuned_model = tuner.hypermodel.build(best_hps)
best_tuned_model.summary()

fit_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

fit_reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

history_2 = best_tuned_model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=[fit_early_stop, fit_reduce_lr],
    verbose=1
)

### Крок 6. Фінальне оцінювання та графіки

Оцініть найкращу модель на **тестовій вибірці**. Наведіть тестові loss і accuracy. Побудуйте графіки навчальної/валідаційної функції втрат і точності.

In [ ]:
test_loss_2, test_acc_2 = best_tuned_model.evaluate(x_test_norm, y_test, verbose=0)

print(f"Найкраща модель Keras Tuner -> Test Loss: {test_loss_2:.4f}, Test Accuracy: {test_acc_2:.4f}")
print(f"Кількість епох навчання (Завдання 1): {len(history_1.history['loss'])}")
print(f"Кількість епох навчання (Завдання 2): {len(history_2.history['loss'])}")

epochs_range_2 = range(1, len(history_2.history['loss']) + 1)

plt.figure(figsize=(14, 5))

# Графік функції втрат
plt.subplot(1, 2, 1)
plt.plot(epochs_range_2, history_2.history['loss'], marker='o', label='Навчальна втрата (loss)')
plt.plot(epochs_range_2, history_2.history['val_loss'], marker='s', label='Валідаційна втрата (val_loss)')
plt.title('Динаміка функції втрат (Keras Tuner)')
plt.xlabel('Епоха')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Графік точності
plt.subplot(1, 2, 2)
plt.plot(epochs_range_2, history_2.history['accuracy'], marker='o', label='Навчальна точність (accuracy)')
plt.plot(epochs_range_2, history_2.history['val_accuracy'], marker='s', label='Валідаційна точність (val_accuracy)')
plt.title('Динаміка точності класифікації (Keras Tuner)')
plt.xlabel('Епоха')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Висновки. Порівняння моделей

| Показник | Модель із завдання 1 | Модель після Keras Tuner |
|---|---|---|
| Кількість прихованих шарів | 2 | `<BEST_NUM_LAYERS>` |
| Кількість нейронів | 256, 128 | `<BEST_UNITS>` |
| Швидкість навчання | 0.001 | `<BEST_LR>` |
| Точність на тестовій вибірці (accuracy) | `<TEST_ACC_1>` | `<TEST_ACC_2>` |
| Втрати на тестовій вибірці (loss) | `<TEST_LOSS_1>` | `<TEST_LOSS_2>` |
| Кількість епох навчання | `<EPOCHS_1>` | `<EPOCHS_2>` |

**Власні висновки:**  
1. У **завданні 1** цільового порогу точності **не нижче 0,87**[cite: 4] успішно досягнуто: базова модель показала на тестовій вибірці `accuracy = <TEST_ACC_1>` (`loss = <TEST_LOSS_1>`).
2. Після автоматизованого підбору гіперпараметрів за допомогою `Keras Tuner (RandomSearch)`[cite: 4] налаштована модель отримала на тесті `accuracy = <TEST_ACC_2>` та `loss = <TEST_LOSS_2>` (порівняй із `<TEST_ACC_1>` — чи зросла точність, чи залишилася на тому ж рівні через обмеження `max_trials = 3`[cite: 4]).
3. Найкращою комбінацією виявилася конфігурація з `<BEST_NUM_LAYERS>` прихованими шарами (`<BEST_UNITS>` нейронів) та швидкістю навчання `<BEST_LR>`; між окремими запусками результати можуть дещо відрізнятися через стохастичну природу алгоритму `RandomSearch`, випадкову ініціалізацію ваг нейронної мережі, перемішування міні-батчів та роботу шарів `Dropout`.
4. Найскладніше серед усіх 10 категорій `Fashion MNIST` класифікувати клас **`Shirt (Сорочка)`**, а також суміжні з ним класи верхнього одягу (`T-shirt/top`, `Pullover`, `Coat`), оскільки на низькій роздільній здатності $28 \times 28$ у відтінках сірого їхні силуети та текстури сильно перекриваються.
5. Загалом застосування `EarlyStopping` та `ReduceLROnPlateau` дозволило уникнути перенавчання в обох експериментах, а ізоляція тестової вибірки до фінального кроку забезпечила об’єктивну оцінку узагальнювальної здатності моделей[cite: 4].

---
**Перед поданням перевірте:** заповнені всі комірки; результати виконання збережені; є графіки, класифікаційний звіт, результати двох моделей і власні висновки. Шаблон не містить готових розв’язків.